<a href="https://colab.research.google.com/github/ayagomaa-cmd/Hotel-Booking-/blob/main/Hotel_Booking_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns







In [3]:
# read the file
df = pd.read_csv('hotel_bookings.csv')
print("Initial Shape:", df.shape)
df.info()

Initial Shape: (119390, 32)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  i

In [4]:
# Count duplicates
print("Exact Duplicates:", df.duplicated().sum())



Exact Duplicates: 31994


In [5]:
#  missing values
missing = df.isnull().sum()
print("\nMissing Values:\n", missing[missing > 0])




Missing Values:
 children         4
country        488
agent        16340
company     112593
dtype: int64


In [6]:
cat_cols = ['hotel', 'meal', 'market_segment', 'distribution_channel', 'reservation_status']
for col in cat_cols:
    print(f"{col}: {df[col].unique()}")

hotel: ['Resort Hotel' 'City Hotel']
meal: ['BB' 'FB' 'HB' 'SC' 'Undefined']
market_segment: ['Direct' 'Corporate' 'Online TA' 'Offline TA/TO' 'Complementary' 'Groups'
 'Undefined' 'Aviation']
distribution_channel: ['Direct' 'Corporate' 'TA/TO' 'Undefined' 'GDS']
reservation_status: ['Check-Out' 'Canceled' 'No-Show']


In [7]:
# Remove  duplicates
df_clean = df.drop_duplicates().copy()


df_clean['children'] = df_clean['children'].fillna(0)
df_clean['country'] = df_clean['country'].fillna('Unknown')
df_clean['agent'] = df_clean['agent'].fillna(0)
df_clean['company'] = df_clean['company'].fillna(0)

In [8]:
# feature Engineering
# Convert float columns to integer
for col in ['children', 'agent', 'company']:
    df_clean[col] = df_clean[col].astype(int)



In [9]:
# standerd date time
df_clean['reservation_status_date'] = pd.to_datetime(df_clean['reservation_status_date'])

# unified arrival_date column
df_clean['arrival_date'] = pd.to_datetime(
    df_clean['arrival_date_year'].astype(str) + '-' +
    df_clean['arrival_date_month'] + '-' +
    df_clean['arrival_date_day_of_month'].astype(str),
    format='%Y-%B-%d'
)

In [10]:
# Anomlies

df_clean['meal'] = df_clean['meal'].replace('Undefined', 'SC')




In [11]:
#Drop the 5 corrupted rows with Undefined market
df_clean = df_clean[
    (df_clean['market_segment'] != 'Undefined') &
    (df_clean['distribution_channel'] != 'Undefined')
]

In [12]:
# create total guests and total total nights
df_clean['total_guests'] = df_clean['adults'] + df_clean['children'] + df_clean['babies']
df_clean['total_nights'] = df_clean['stays_in_weekend_nights'] + df_clean['stays_in_week_nights']

/tmp/ipykernel_1440/2066933007.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['total_guests'] = df_clean['adults'] + df_clean['children'] + df_clean['babies']
/tmp/ipykernel_1440/2066933007.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['total_nights'] = df_clean['stays_in_weekend_nights'] + df_clean['stays_in_week_nights']


In [13]:
# outliers
df_clean = df_clean[df_clean['total_guests'] > 0]  # negative prices
df_clean = df_clean[(df_clean['adr'] >= 0) & (df_clean['adr'] < 5000)] # Extremly high prices

In [14]:
# row filtering
df_clean = df_clean.reset_index(drop=True)
print("Final Cleaned Shape:", df_clean.shape)

Final Cleaned Shape: (87223, 35)


In [15]:
print("Statistical Summary of Key Features:")
display(df_clean[['adr', 'total_guests', 'total_nights', 'lead_time']].describe())

Statistical Summary of Key Features:


,adr,total_guests,total_nights,lead_time
count,87223.000000,87223.000000,87223.000000,87223.000000
mean,106.462091,2.029086,3.628504,79.973482
std,51.880379,0.790159,2.742906,86.059504
min,0.000000,1.000000,0.000000,0.000000
25%,72.250000,2.000000,2.000000,11.000000
50%,98.200000,2.000000,3.000000,49.000000
75%,134.100000,2.000000,5.000000,125.000000
max,510.000000,55.000000,69.000000,737.000000


In [19]:
df_clean.to_csv('hotel_bookings_cleaned.csv', index=False)

In [22]:
assert df_clean.shape == (87223, 35), f"Unexpected shape: {df_clean.shape}"
assert df_clean.isnull().sum().sum() == 0, "Null values still exist"
assert df_clean.duplicated().sum() == 0, "Duplicate rows still exist"
print("All structural integrity checks passed.")

All structural integrity checks passed.
